# FGD Multimodal Master's Thesis Pipeline — Colab (T4-safe)

**What changed vs. your previous version (and why it kept failing):**

1. **Removed the Python 3.10 / Miniconda downgrade.** The `wget -qO miniconda.sh https://anaconda.com` line was not a real installer URL, and rebuilding a parallel Python inside a live Colab runtime is fragile and unnecessary — Colab's stock runtime already works fine with current `transformers`/`bitsandbytes`/`torch` wheels.
2. **Stopped reinstalling PyTorch.** `--index-url https://pytorch.org` isn't a valid package index, and reinstalling torch/vision/audio on top of Colab's preinstalled, GPU-matched build is what triggered the long "Preparing metadata (pyproject.toml)" source-build hang you interrupted (that `KeyboardInterrupt` in your log). We only install the extra libraries actually needed.
3. **Split the giant single cell into separate cells.** Your version concatenated install → transcription → vision → LLM → packaging into one block with indentation errors (`def ask(...)` body, the chunking `for` loop) introduced by the copy/paste. One cell per stage means you can see exactly where something fails and re-run just that stage without repeating multi-minute steps.
4. **Added explicit checks and cleanup** (file-exists asserts, GPU check, VRAM cleanup between models) so failures give a clear message instead of a wall of traceback.
5. Kept your actual pipeline logic (ffmpeg mastering → faster-whisper large-v3 → Qwen2.5-VL-3B frame captions → Qwen2.5-3B-Instruct thematic report → docx/zip package) unchanged — that part of your design was fine.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Disconnect and delete runtime` if you've run the broken version before, so you start clean. Run the cells **in order**, top to bottom — don't run everything as one cell.


### Step 0 — Confirm GPU and environment

In [1]:
import sys, torch

print(f"Python: {sys.version.split()[0]}")
assert torch.cuda.is_available(), (
    "No GPU visible. Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU, "
    "then re-run this cell."
)
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)


Python: 3.13.15
GPU: Tesla T4
Torch: 2.11.0+cu128 | CUDA: 12.8


### Step 1 — Install only what's missing

We deliberately do **not** touch `torch`/`torchvision`/`torchaudio` — Colab's T4 runtime ships a
CUDA-matched build already, and reinstalling it is what caused the previous hang. `ffmpeg` is
already present on Colab too; the `apt-get` call is just a safety net.

In [2]:
%%capture install_log
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q -U "transformers>=4.49.0" accelerate bitsandbytes
!pip install -q -U faster-whisper librosa soundfile python-docx qwen-vl-utils


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Sanity check the installs actually landed (run right after the cell above)
import importlib
for pkg in ["transformers", "accelerate", "bitsandbytes", "faster_whisper",
            "librosa", "soundfile", "docx", "qwen_vl_utils"]:
    mod = importlib.import_module(pkg)
    print(f"{pkg:16s} OK  ({getattr(mod, '__version__', 'n/a')})")


transformers     OK  (5.17.0)
accelerate       OK  (1.15.0)
bitsandbytes     OK  (0.50.2)
faster_whisper   OK  (1.2.1)
librosa          OK  (1.0.0)
soundfile        OK  (0.14.0)
docx             OK  (1.2.0)
qwen_vl_utils    OK  (n/a)


### Step 2 — Google Drive + (optional) Hugging Face token

In [5]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive', force_remount=True)

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab secrets.")
except Exception:
    print("No HF_TOKEN secret set — continuing unauthenticated (fine for the public models used here).")


Mounted at /content/drive
HF_TOKEN loaded from Colab secrets.


### Step 3 — Point at your source file and master the audio

Update `SOURCE` to the real path of your recording in Drive before running.

In [6]:
import subprocess, json
from pathlib import Path

SOURCE = '/content/drive/MyDrive/FGD/2023-12-19 12-56-21.mp4'  # <-- update this path

ROOT = Path("/content/fgd_thesis")
ROOT.mkdir(exist_ok=True)

assert Path(SOURCE).exists(), f"Source file not found: {SOURCE} (check the path/spelling in Drive)"

suffix = Path(SOURCE).suffix.lower()
assert suffix in {".mp4", ".mov", ".mkv", ".avi", ".mp3", ".wav", ".m4a", ".aac", ".flac", ".ogg", ".opus"}, \
    f"Unsupported file type: {suffix}"
HAS_VIDEO = suffix in {".mp4", ".mov", ".mkv", ".avi"}
print("Source:", SOURCE, "| contains video:", HAS_VIDEO)

AUDIO = "/content/audio.wav"
FILTER = (
    "highpass=f=70,lowpass=f=7600,afftdn=nr=10:nf=-35:tn=1,"
    "acompressor=threshold=-20dB:ratio=2.5:attack=20:release=250,"
    "silenceremove=stop_periods=-1:stop_duration=1.2:stop_threshold=-38dB:stop_silence=0.35,"
    "loudnorm=I=-16:TP=-1.5:LRA=11"
)

subprocess.run(
    ["ffmpeg", "-hide_banner", "-y", "-i", SOURCE, "-vn", "-af", FILTER,
     "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", AUDIO],
    check=True,
)

probe = json.loads(subprocess.check_output(
    ["ffprobe", "-v", "error", "-show_entries",
     "format=duration,size:stream=codec_type,codec_name,width,height,sample_rate,channels",
     "-of", "json", SOURCE], text=True))
audio_probe = json.loads(subprocess.check_output(
    ["ffprobe", "-v", "error", "-show_entries",
     "format=duration,size:stream=sample_rate,channels", "-of", "json", AUDIO], text=True))

processing = {
    "source": SOURCE,
    "source_probe": probe,
    "audio_output": "audio.wav",
    "audio_probe": audio_probe,
    "audio_processing": (
        "70 Hz high-pass; 7.6 kHz low-pass; gentle denoise; compression; "
        "-16 LUFS target; -1.5 dBTP target; silences longer than 1.2s reduced to 0.35s."
    ),
    "timestamp_rule": "Transcript times map to the compacted audio.wav, not the original video.",
}
print(json.dumps(processing, indent=2))


Source: /content/drive/MyDrive/FGD/2023-12-19 12-56-21.mp4 | contains video: True
{
  "source": "/content/drive/MyDrive/FGD/2023-12-19 12-56-21.mp4",
  "source_probe": {
    "programs": [],
    "streams": [
      {
        "codec_name": "h264",
        "codec_type": "video",
        "width": 1920,
        "height": 1080
      },
      {
        "codec_name": "aac",
        "codec_type": "audio",
        "sample_rate": "44100",
        "channels": 1
      }
    ],
    "format": {
      "duration": "2281.766667",
      "size": "886645154"
    }
  },
  "audio_output": "audio.wav",
  "audio_probe": {
    "programs": [],
    "streams": [
      {
        "sample_rate": "16000",
        "channels": 1
      }
    ],
    "format": {
      "duration": "1675.328500",
      "size": "53610590"
    }
  },
  "audio_processing": "70 Hz high-pass; 7.6 kHz low-pass; gentle denoise; compression; -16 LUFS target; -1.5 dBTP target; silences longer than 1.2s reduced to 0.35s.",
  "timestamp_rule": "Transcri

### Step 4 — Transcribe with faster-whisper (large-v3, float16 on the T4)

In [7]:
import gc, torch, pandas as pd
from faster_whisper import WhisperModel

print("Loading Whisper large-v3 onto the GPU...")
whisper = WhisperModel("large-v3", device="cuda", compute_type="float16")

print("Transcribing...")
segs, info = whisper.transcribe(
    AUDIO, beam_size=5, best_of=5, temperature=0,
    word_timestamps=True, vad_filter=False, condition_on_previous_text=True,
)

rows = [
    {
        "segment_id": i,
        "compact_start": round(s.start, 2),
        "compact_end": round(s.end, 2),
        "duration": round(s.end - s.start, 2),
        "speaker": "UNVERIFIED_SPEAKER",
        "text": s.text.strip(),
        "avg_logprob": s.avg_logprob,
    }
    for i, s in enumerate(segs, 1)
]
df = pd.DataFrame(rows)
assert not df.empty, "No speech was transcribed — check the source audio has audible speech."

def stamp(t):
    return f"{int(t // 3600):02}:{int(t % 3600 // 60):02}:{int(t % 60):02}"

transcript = "\n".join(
    f"[{stamp(r.compact_start)}\u2013{stamp(r.compact_end)}] {r.speaker}: {r.text}"
    for r in df.itertuples()
)

(ROOT / "transcript_timestamped.txt").write_text(transcript, encoding="utf-8")
df.to_csv(ROOT / "transcription.csv", index=False, encoding="utf-8-sig")

print("Language:", info.language, "| confidence:", round(info.language_probability, 3), "| segments:", len(df))

# Free VRAM before loading the vision model
del whisper
gc.collect()
torch.cuda.empty_cache()


Loading Whisper large-v3 onto the GPU...
Transcribing...
Language: ar | confidence: 0.993 | segments: 160


### Step 5 — Sample video frames and caption them (Qwen2.5-VL-3B, 4-bit)

Skipped automatically if the source is audio-only.

In [8]:
import numpy as np, subprocess, gc, torch, json

frames = []
if HAS_VIDEO:
    frame_dir = ROOT / "video_frames"
    frame_dir.mkdir(exist_ok=True)
    duration = float(probe["format"]["duration"])
    times = np.linspace(0, duration, 20, endpoint=False) + min(5, duration / 40)
    for n, t in enumerate(times, 1):
        out = frame_dir / f"frame_{n:02d}_{int(t):04d}s.jpg"
        subprocess.run(
            ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-ss", str(round(float(t), 2)),
             "-i", SOURCE, "-frames:v", "1", "-vf", "scale='min(960,iw)':-2", str(out)],
            check=True,
        )
        frames.append({"original_video_time_seconds": round(float(t), 2), "file": str(out)})
print("Frames sampled:", len(frames))

visual = []
if frames:
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
    from qwen_vl_utils import process_vision_info

    qconf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
    vl = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2.5-VL-3B-Instruct", quantization_config=qconf, device_map="auto"
    )

    visual_prompt = (
        "This is a sampled frame from a focus-group recording. Describe only plainly observable, "
        "research-relevant visual facts: setting, seating/visibility, materials, apparent turn-taking "
        "or group interaction, technical/recording issues. Do not identify people, infer demographics, "
        "emotions, intent, relationships, or sensitive traits. State uncertainty and write 3\u20136 short "
        "evidence-log bullets."
    )

    for item in frames:
        msg = [{"role": "user", "content": [
            {"type": "image", "image": item["file"]},
            {"type": "text", "text": visual_prompt},
        ]}]
        text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        ii, vi = process_vision_info(msg)
        inp = processor(text=[text], images=ii, videos=vi, padding=True, return_tensors="pt").to("cuda")
        ids = vl.generate(**inp, max_new_tokens=280, do_sample=False)
        item["observation"] = processor.batch_decode(ids[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0]
        visual.append(item)

    del vl, processor
    gc.collect()
    torch.cuda.empty_cache()

(ROOT / "visual_evidence_log.json").write_text(json.dumps(visual, ensure_ascii=False, indent=2), encoding="utf-8")
print("Visual evidence entries:", len(visual))


Frames sampled: 20


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Visual evidence entries: 20


### Step 6 — Audio technical quality notes

In [9]:
import librosa, numpy as np, json

y, sr = librosa.load(AUDIO, sr=16000, mono=True)
voice_note = {
    "compact_duration_seconds": round(len(y) / sr, 2),
    "sample_rate": sr,
    "channels": 1,
    "mean_rms": round(float(np.sqrt(np.mean(y ** 2))), 5),
    "estimated_speech_segments": len(df),
    "interpretation": (
        "Technical recording indicators only. Do not treat these measures as participant "
        "traits, emotion, credibility, or diagnosis."
    ),
}
(ROOT / "audio_voice_technical_notes.json").write_text(json.dumps(voice_note, indent=2), encoding="utf-8")
print(json.dumps(voice_note, indent=2))


{
  "compact_duration_seconds": 1675.33,
  "sample_rate": 16000,
  "channels": 1,
  "mean_rms": 0.13958,
  "estimated_speech_segments": 160,
  "interpretation": "Technical recording indicators only. Do not treat these measures as participant traits, emotion, credibility, or diagnosis."
}


### Step 7 — Thematic analysis and report synthesis (Qwen2.5-3B-Instruct, 4-bit)

In [ ]:
import gc, torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

qconf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
llm = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct", quantization_config=qconf, device_map="auto"
)

# def ask(prompt, n=1700):
#     msgs = [
#         {"role": "system", "content": (
#             "You are a master's-level qualitative research assistant. Use supplied evidence only. "
#             "Separate transcript evidence, machine observations, and researcher interpretation. "
#             "Never invent quotations, speakers, identities, demographics, causality, consensus, "
#             "or visual/emotional facts."
#         )},
#         {"role": "user", "content": prompt},
#     ]
#     x = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(llm.device)
#     z = llm.generate(x, max_new_tokens=n, do_sample=False, pad_token_id=tok.eos_token_id)
#     return tok.decode(z[0][x.shape[1]:], skip_special_tokens=True)
def ask(prompt, n=1700):
    msgs = [
        {"role": "system", "content": (
            "You are a master's-level qualitative research assistant. Use supplied evidence only. "
            "Separate transcript evidence, machine observations, and researcher interpretation. "
            "Never invent quotations, speakers, identities, demographics, causality, consensus, "
            "or visual/emotional facts."
        )},
        {"role": "user", "content": prompt},
    ]
    inputs = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(llm.device)
    input_len = inputs["input_ids"].shape[1]
    z = llm.generate(**inputs, max_new_tokens=n, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(z[0][input_len:], skip_special_tokens=True)

lines = transcript.splitlines()
chunks, cur, length = [], [], 0
for line in lines:
    if length + len(line) > 8500 and cur:
        chunks.append("\n".join(cur))
        cur, length = [], 0
    cur.append(line)
    length += len(line) + 1
if cur:
    chunks.append("\n".join(cur))

print(f"Analyzing {len(chunks)} transcript section(s)...")
part = [
    ask(
        "Analyse this transcript section. Return: questions asked; verbatim-grounded answers; "
        "provisional themes/subthemes; agreement/disagreement; counterexamples; timestamped evidence; "
        "data-quality cautions.\n\n" + c
    )
    for c in chunks
]

visual_text = json.dumps(visual, ensure_ascii=False) if visual else "No video frames were supplied."

report_prompt = (
    "Create a comprehensive, thesis-ready Focus Group Discussion analytic report from the "
    "transcript analyses and visual evidence log below. Required headings:\n\n"
    "1. Study/Session Metadata (use placeholders where unknown)\n"
    "2. Data Sources and Processing Audit Trail\n"
    "3. Transcription and Speaker-Attribution Limitations\n"
    "4. Audio/Voice Technical Quality Notes\n"
    "5. Visual Evidence Log (only observable facts; mark machine-generated)\n"
    "6. Researcher Field Notes and Reflexive Notes Template\n"
    "7. Question\u2013Answer Matrix (question, evidence-based answer summary, dissent/caveat, timestamps)\n"
    "8. Thematic Analysis: theme, definition, subthemes, supporting evidence, negative case, interpretation\n"
    "9. Agreement/Disagreement and Interaction Patterns\n"
    "10. Preliminary Codebook (code, definition, include/exclude, example, related theme)\n"
    "11. Findings Narrative\n"
    "12. Trustworthiness: credibility, dependability, confirmability, transferability\n"
    "13. Ethics, anonymisation, and data-management notes\n"
    "14. Master's Thesis Reporting Checklist\n"
    "15. Limitations and Required Human Validation.\n\n"
    "Never present automated inference as fact. Do not name participants, infer traits, or "
    "claim video findings beyond the evidence log.\n\n"
    "TRANSCRIPT ANALYSES:\n" + "\n\n---\n".join(part) +
    "\n\nVISUAL EVIDENCE LOG:\n" + visual_text +
    "\n\nAUDIO TECHNICAL NOTE:\n" + json.dumps(voice_note)
)

# report = ask(report_prompt, 3000)
# (ROOT / "FGD_comprehensive_thesis_report.md").write_text(report, encoding="utf-8")
# print(report[:1500])
# print("\n... [truncated in output — full report saved to FGD_comprehensive_thesis_report.md]")
import gc, torch

def ask_safe(prompt, n):
    """Retry with a shrinking token budget instead of dying on one big OOM."""
    for attempt_n in [n, max(1000, n // 2), max(600, n // 3)]:
        gc.collect()
        torch.cuda.empty_cache()
        try:
            return ask(prompt, attempt_n)
        except torch.cuda.OutOfMemoryError:
            print(f"OOM at max_new_tokens={attempt_n}, retrying with a smaller budget...")
    raise RuntimeError(
        "Still OOM even at a reduced token budget. See the two-pass split below."
    )

# report = ask_safe(report_prompt, 1400)  # was 3000 — that's what tipped it over on a T4
# (ROOT / "FGD_comprehensive_thesis_report.md").write_text(report, encoding="utf-8")
# print(report[:1500])
# print("\n... [truncated in output — full report saved to FGD_comprehensive_thesis_report.md]")
report_prompt_1 = report_prompt + "\n\nFor this pass, write ONLY sections 1–8 of the required headings."
report_prompt_2 = report_prompt + "\n\nFor this pass, write ONLY sections 9–15 of the required headings."

part1 = ask_safe(report_prompt_1, 900)
gc.collect(); torch.cuda.empty_cache()
part2 = ask_safe(report_prompt_2, 900)

report = part1 + "\n\n" + part2
(ROOT / "FGD_comprehensive_thesis_report.md").write_text(report, encoding="utf-8")
print(report[:1500])

del llm, tok
gc.collect()
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Analyzing 3 transcript section(s)...


In [18]:
torch.cuda.empty_cache()
gc.collect()
!nvidia-smi


Fri Sep 11 20:23:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             32W /   70W |    5899MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 8 — Package everything into a downloadable zip

In [ ]:
import shutil, json
from pathlib import Path
from docx import Document
from google.colab import files

OUT = Path("/content/FGD_THESIS_PACKAGE")
OUT.mkdir(exist_ok=True)

for p in [
    Path(AUDIO), ROOT / "transcript_timestamped.txt", ROOT / "transcription.csv",
    ROOT / "visual_evidence_log.json", ROOT / "audio_voice_technical_notes.json",
    ROOT / "FGD_comprehensive_thesis_report.md",
]:
    shutil.copy(p, OUT / p.name)

(OUT / "processing_audit_trail.json").write_text(json.dumps(processing, indent=2), encoding="utf-8")

researcher_template = '''# Researcher Notes and Validation Template
Session ID: ____  Date/time: ____  Facilitator: ____  Research question(s): ____

## Context and reflexivity
- Recruitment/context that may shape discussion:
- Facilitator actions or prompts:
- Researcher assumptions/positionality:
- Technical disruptions and their likely effect:

## Validation log
- Transcript passages corrected (timestamp / original / revised / reviewer):
- Speaker labels verified (timestamp / label / evidence):
- Visual observation verified or rejected (video time / observation / reviewer):
- Theme/code changed after review (old / new / rationale):
- Negative cases and dissent retained:

## Analytic memo
- Emerging explanation:
- Competing explanation:
- Evidence needed before thesis claim:
- Quote candidates checked against source:
'''
(OUT / "researcher_notes_template.md").write_text(researcher_template, encoding="utf-8")

doc = Document()
doc.add_heading("Focus Group Discussion \u2014 Comprehensive Thesis Analysis", 0)
for line in report.splitlines():
    if line.startswith("#"):
        doc.add_heading(line.lstrip("# ").strip(), min(3, len(line) - len(line.lstrip("#"))))
    elif line.strip():
        doc.add_paragraph(line)
doc.add_page_break()
doc.add_heading("Timestamped Transcript", 1)
for line in transcript.splitlines():
    doc.add_paragraph(line)
doc.save(OUT / "FGD_comprehensive_thesis_report.docx")

zip_path = shutil.make_archive("/content/FGD_THESIS_PACKAGE", "zip", OUT)
print("Package ready:", zip_path)
print(*sorted(p.name for p in OUT.iterdir()), sep="\n")

files.download(zip_path)
